## Business Understanding

## Imports

In [146]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from nltk.stem import WordNetLemmatizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...


## Data Understanding

In [147]:
df = pd.read_csv('../data/judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [148]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [149]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [150]:
df.shape

(9093, 3)

## Data Preparation
1. **Lemmatization**- examines the morphology of the words and reduces each word to its most basic form or lemma eg studies become study.

2. **Lowercasing**- reduces the number of unique words words the model must handle, improves model consistency and ensures the model learns **all occurences of a word together**, and simplifies **text processing**

3. **Tokenization and removing stopwords**-splits the sentences into array of individual words. Stopwords(common words or pretty useless data) that contain litte or no information.

4. **Label Encoding**-a simple and very common technique in Machine Learning to convert **categorical labels**(text) into **numbers**.

5. **Stemming**- stemming refers to the removal of suffixes eg cats to cat.

In [151]:
df.isnull().sum() ## check for null entries



tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [152]:
# dealing with the missing values. Do we drop of fill?
# for tweet, we will drop that one missing value

df = df.dropna(subset=['tweet_text']) # dropped the one missing tweet
df.isnull().sum() 

tweet_text                                               0
emotion_in_tweet_is_directed_at                       5801
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [153]:
# Duplicates
duplicates = df['tweet_text'].duplicated().sum()  # check for duplicates in the 'tweet' column
print(f"Duplicate tweets: {duplicates}")
df = df.drop_duplicates(subset='tweet_text').reset_index(drop=True) # drop duplicates based on the 'tweet' column and reset index
print(f"Shape after removing duplicates: {df.shape}")

Duplicate tweets: 27
Shape after removing duplicates: (9065, 3)


In [154]:
## Normalizing sentiment labels and removing "I can't tell" entries
df['emotion_in_tweet_is_directed_at'] = df['emotion_in_tweet_is_directed_at'].str.lower().str.strip()
df = df[df['emotion_in_tweet_is_directed_at'] != "i can't tell"]

print(f"Shape after removing ambiguous labels: {df.shape}")
print(df['is_there_an_emotion_directed_at_a_brand_or_product'].value_counts())

Shape after removing ambiguous labels: (9065, 3)
No emotion toward brand or product    5372
Positive emotion                      2968
Negative emotion                       569
I can't tell                           156
Name: is_there_an_emotion_directed_at_a_brand_or_product, dtype: int64


## Text Cleaning Function

In [155]:
## Mapping the product names to their respective brands

apple_list = [x.lower() for x in ['iPad', 'Apple', 'iPad app', 'iPhone app', 'iPhone', 'Mac', 'MacBook']]
google_list = [x.lower() for x in ['Google' , 'Android App', 'Android']]

def map_to_brand(product_name):
    if pd.isna(product_name):
        return None
    
    product_name = str(product_name).lower()
    
    if product_name in apple_list:
        return 'Apple'
    elif product_name in google_list:
        return 'Google'
    else:
        return None

df['product'] = df['emotion_in_tweet_is_directed_at'].apply(map_to_brand)

def fill_missing_product(row):
    if pd.notna(row['product']):
        return row['product']
    
    text = str(row['tweet_text']).lower()
    
    if any(word in text for word in ['apple', 'ipad', 'iphone', 'mac']):
        return 'Apple'
    elif any(word in text for word in ['google', 'android', 'pixel']):
        return 'Google'
    else:
        return 'Not specified'

df['product'] = df.apply(fill_missing_product, axis=1)

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def get_clean_tokens(text):
    if not isinstance(text, str):
        return []
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtag symbols (keep the word)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stop words
    
    tokens = [word for word in tokens if word not in stop_words]
    # Remove short tokens (1-2 chars)
    tokens = [t for t in tokens if len(t) > 2]
    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

In [157]:
## Apply the cleaning function to the 'tweet_text' column and create a new column 'clean_text'

df['clean_text'] = df['tweet_text'].apply(lambda x: " ".join(get_clean_tokens(x)))

df[['tweet_text', 'clean_text']].head()

,tweet_text,clean_text
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iphone hr tweeting riseaustin dead need upgrad...
1,@jessedee Know about @fludapp ? Awesome iPad/i...,know awesome ipadiphone app youll likely appre...
2,@swonderlin Can not wait for #iPad 2 also. The...,wait ipad also sale sxsw
3,@sxsw I hope this year's festival isn't as cra...,hope year festival isnt crashy year iphone app...
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,great stuff fri sxsw marissa mayer google tim ...


In [158]:
positive_emotion = []
negative_emotion = []


for text, is_there_an_emotion_directed_at_a_brand_or_product in zip(df['tweet_text'], df['is_there_an_emotion_directed_at_a_brand_or_product']):
    tokens = get_clean_tokens(text)
    if is_there_an_emotion_directed_at_a_brand_or_product == 'Positive emotion':
        positive_emotion.extend(tokens)

    elif is_there_an_emotion_directed_at_a_brand_or_product == 'Negative emotion':
        negative_emotion.extend(tokens)

# Top 15 Positive tweets
print("\n=== Top 15 Words in POSITIVE Sentiment ===")
print(Counter(positive_emotion).most_common(15))

# Top 15 Negative tweets
print("\n=== Top 15 Words in NEGATIVE Sentiment ===")
print(Counter(negative_emotion).most_common(15))


=== Top 15 Words in POSITIVE Sentiment ===
[('sxsw', 3102), ('link', 1210), ('ipad', 1191), ('apple', 879), ('google', 690), ('store', 548), ('iphone', 522), ('app', 394), ('new', 359), ('austin', 290), ('popup', 218), ('android', 197), ('get', 181), ('amp', 178), ('launch', 174)]

=== Top 15 Words in NEGATIVE Sentiment ===
[('sxsw', 578), ('ipad', 194), ('iphone', 155), ('google', 141), ('apple', 107), ('link', 102), ('app', 60), ('store', 45), ('new', 43), ('like', 42), ('circle', 36), ('need', 35), ('social', 30), ('apps', 30), ('people', 29)]


## TF -IDF

TF - Maps out the frequency of tokens in our data
IDF - shows the relevance/Weight of words to our target variable

In [159]:
## checking the total vocabulary size
temp_vectorizer = TfidfVectorizer()
temp_vectorizer.fit(df['clean_text'])
vocab_size = len(temp_vectorizer.vocabulary_)
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 9256


In [160]:
tfidf = TfidfVectorizer(max_features=10000)

X = tfidf.fit_transform(df['clean_text'])
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

In [161]:
print(X)

  (0, 7914)	0.04684460922332671
  (0, 7670)	0.34978938361923495
  (0, 5908)	0.39407358029926953
  (0, 8651)	0.3463547706363212
  (0, 5200)	0.21703588441577187
  (0, 1935)	0.33470733550779425
  (0, 6883)	0.41774833847174225
  (0, 8514)	0.3176647385070598
  (0, 3741)	0.38562369335956714
  (0, 4083)	0.13012911675627872
  (1, 3012)	0.2008056068342442
  (1, 3224)	0.2763628942699317
  (1, 8202)	0.2986212929217139
  (1, 204)	0.27318331591186473
  (1, 2027)	0.24911483976499624
  (1, 377)	0.3940429277458601
  (1, 4530)	0.353916655564587
  (1, 9205)	0.3091889094520307
  (1, 333)	0.164632258114974
  (1, 4062)	0.36985128396307554
  (1, 572)	0.24838059359493384
  (1, 4341)	0.23116247392171882
  (1, 7914)	0.047867356921606076
  (2, 6977)	0.5907860824875228
  (2, 4051)	0.21006058386881324
  :	:
  (9062, 5231)	0.18905588432660608
  (9062, 5993)	0.22086879817098715
  (9062, 9192)	0.17426372177208815
  (9062, 3275)	0.0685127066310618
  (9062, 7914)	0.02926885543608262
  (9063, 9200)	0.38514492843839054


In [162]:
print(y)

0                         Negative emotion
1                         Positive emotion
2                         Positive emotion
3                         Negative emotion
4                         Positive emotion
                       ...                
9060                      Positive emotion
9061    No emotion toward brand or product
9062    No emotion toward brand or product
9063    No emotion toward brand or product
9064    No emotion toward brand or product
Name: is_there_an_emotion_directed_at_a_brand_or_product, Length: 9065, dtype: object


## Train/Test Split

In [163]:

X_raw = df['clean_text']
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Fit TF-IDF on training data only and transform both train and test sets
tfidf = TfidfVectorizer(max_features=10000)
X_train = tfidf.fit_transform(X_train_raw)   
X_test  = tfidf.transform(X_test_raw)         

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")

Train shape: (7252, 8326)
Test shape:  (1813, 8326)


## Modelling
The goal of this section is to build and compare classification models that can accurately predict sentiment from tweet text.

We begin with a baseline model, then introduce more complex models to improve performance.

## Baseline Model
### Logistic Regresison
Logistic Regression was chosen as a simple and interpretable linear model suitable for text classification.

In [172]:
log_model = LogisticRegression(max_iter=5000, class_weight='balanced')
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

In [173]:
print(classification_report(y_test, y_pred_log))

                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       0.36      0.65      0.46       114
No emotion toward brand or product       0.78      0.65      0.71      1074
                  Positive emotion       0.57      0.63      0.59       594

                          accuracy                           0.63      1813
                         macro avg       0.43      0.48      0.44      1813
                      weighted avg       0.67      0.63      0.64      1813



### Decision Trees
A Decision Tree model was introduced to capture non-linear relationships between features.

In [171]:
tree_model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=20,
    min_samples_leaf=5
)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

print("=== Decision Tree ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

=== Decision Tree ===
Accuracy: 0.380584666298952
                                    precision    recall  f1-score   support

                      I can't tell       0.04      0.10      0.05        31
                  Negative emotion       0.12      0.68      0.20       114
No emotion toward brand or product       0.73      0.38      0.50      1074
                  Positive emotion       0.40      0.33      0.36       594

                          accuracy                           0.38      1813
                         macro avg       0.32      0.37      0.28      1813
                      weighted avg       0.57      0.38      0.43      1813



### Naive Bayes


In [167]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
print("=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

=== Naive Bayes ===
Accuracy:

 0.646442360728075
                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        31
                  Negative emotion       1.00      0.03      0.05       114
No emotion toward brand or product       0.64      0.95      0.76      1074
                  Positive emotion       0.71      0.25      0.36       594

                          accuracy                           0.65      1813
                         macro avg       0.59      0.31      0.30      1813
                      weighted avg       0.67      0.65      0.58      1813



## Evaluation

In [175]:
# Summary comparison of all models
results = {
    'Logistic Regression': accuracy_score(y_test, y_pred_log),
    'Decision Tree':        accuracy_score(y_test, y_pred_tree),
    'Naive Bayes':          accuracy_score(y_test, y_pred_nb),
}

print("=== Model Accuracy Comparison ===")
for model, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {model:<25} {acc:.4f}")

=== Model Accuracy Comparison ===
  Naive Bayes               0.6464
  Logistic Regression       0.6315
  Decision Tree             0.3806
